# 3b. LSTM Temporal Modeling on DCNN FC7 Features

This notebook trains an LSTM model on top of deep features extracted from a trained DCNN. In simple terms, the DCNN turns each audio segment into a compact numeric description, and the LSTM looks at the whole sequence of those segments to learn how emotions change over time.

What this notebook does:
1. Load segment-level audio features and labels.
2. Extract FC7 features from a pretrained DCNN.
3. Group segments back into utterances.
4. Train an LSTM to predict emotion labels over time.
5. Smooth the predictions so they are less noisy.
6. Compare predictions with the ground truth using both segment-level accuracy and temporal overlap (t-IoU).
7. Summarize how well each emotion is detected.

## Section 1. Importing Libraries and Making Runs Reproducible

### What is happening here?
This section brings in all the tools we need for the notebook. That includes general Python helpers, numerical computing libraries, deep learning tools, and evaluation functions. We also fix the random seeds so that the notebook behaves as consistently as possible from one run to the next.

### Why this matters
When you train a neural network, the results can change slightly each time because the model starts from a random state and the GPU can introduce small differences. By setting seeds and deterministic options, we reduce that randomness and make the results easier to compare.

### Main items used here
- `os` and `csv` for file handling
- `defaultdict` and `Counter` for grouping and voting
- `numpy` for array operations
- `torch` for model training
- `torchvision` for image transforms and AlexNet
- `sklearn.metrics` for accuracy and report tables
- `EMOTION_ENG_MAP` and `EMOTION_CODE_MAP` for mapping emotion IDs and names

### Step-by-step flow
1. Import the libraries needed by the notebook.
2. Import the project-specific emotion label maps.
3. Set the random seed to `42` for NumPy and PyTorch.
4. Turn on deterministic behavior in cuDNN when possible.
5. Keep GPU behavior as stable as the framework allows.

In [ ]:
import os
import csv
from collections import defaultdict, Counter
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import accuracy_score, classification_report

from utility import EMOTION_ENG_MAP, EMOTION_CODE_MAP

np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## Section 2. Global Settings and Experiment Choices

### What is happening here?
This section defines the main switches and constants for the whole notebook. Think of it as the control panel. If you want to change the dataset, switch between original split and LOSO split, or adjust the temporal evaluation settings, this is the place to do it.

### Important parameters
- `DATASET`: the folder containing the preprocessed data
- `SPLIT_MODE`: chooses between `"original"` and `"loso"`
- `NUM_CLASSES = 7`: number of emotion categories
- `HOP_MS = 0.010`: frame hop size in seconds
- `WINDOW_MS = 0.025`: frame window size in seconds
- `SEG_FRAMES = 64`: number of frames in each segment
- `FRAME_SHIFT = 30`: shift between neighboring segments
- `SMOOTH_WINDOW = 5`: size of the majority-vote smoothing window
- `TIOU_THRESHOLD = 0.50`: minimum overlap for a successful match

### Step-by-step flow
1. Read the current working directory.
2. Build the dataset, model, and results paths.
3. Create the results folder if it does not already exist.
4. Choose the compute device (`cuda` if available, otherwise `cpu`).
5. Define the frame and segment timing values used later for temporal evaluation.
6. Find the available folds if LOSO mode is turned on.
7. Print the chosen settings so the run is easy to verify.

In [ ]:
CURR_DIR = os.getcwd()

#===================================================

# Update as needed
DATASET = "processed_emodb_comb_norm_loso"
SPLIT_MODE = "loso"  # "original" or "loso"

#===================================================

DATASET_PATH = os.path.join(CURR_DIR, DATASET)
MODEL_DIR = os.path.join(CURR_DIR, "models")
RESULTS_DIR = os.path.join(CURR_DIR, "results", DATASET)
os.makedirs(RESULTS_DIR, exist_ok=True)

RAW_TIMELINE_ROOT = os.path.abspath(os.path.join(CURR_DIR, "../emo_db_comb"))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 7
HOP_MS = 0.010
WINDOW_MS = 0.025
SEG_FRAMES = 64
FRAME_SHIFT = 30
SEG_HOP_SEC = HOP_MS * FRAME_SHIFT
SEG_DUR_SEC = WINDOW_MS + (SEG_FRAMES - 1) * HOP_MS

SMOOTH_WINDOW = 5
TIOU_THRESHOLD = 0.50

if SPLIT_MODE == "loso":
    FOLD_NAMES = sorted([
        name for name in os.listdir(DATASET_PATH)
        if name.startswith("fold") and os.path.isdir(os.path.join(DATASET_PATH, name))
    ])
    if len(FOLD_NAMES) == 0:
        raise ValueError(f"No fold directories found under {DATASET_PATH}.")
else:
    FOLD_NAMES = []

print(f"DATASET_PATH: {DATASET_PATH}")
print(f"SPLIT_MODE: {SPLIT_MODE}")
if SPLIT_MODE == "loso":
    print(f"Folds: {FOLD_NAMES}")

DATASET_PATH: /Users/leyanzhi/Repositories/SC4001-Neural-Network-Project/SER_DCNN_DTPM/processed_emodb_comb_norm_loso
SPLIT_MODE: loso
Folds: ['fold1', 'fold10', 'fold2', 'fold3', 'fold4', 'fold5', 'fold6', 'fold7', 'fold8', 'fold9']


## Section 3. Turning Each Audio Segment into a Deep Feature

### What is happening here?
Here we prepare the preprocessing pipeline that lets a trained AlexNet-style DCNN turn each audio segment into a 4096-dimensional feature vector. This feature is often called the FC7 representation. It is much smaller than the raw audio image and usually more useful for sequence learning.

### Why this matters
The LSTM does not work directly on raw spectrogram images. Instead, it learns from the compact FC7 vectors produced by the DCNN. This lets the LSTM focus on the emotion pattern over time instead of learning low-level image details from scratch.

### Important parameters
- Input segment shape before preprocessing: `3 x 64 x 64`
- AlexNet input size: `227 x 227`
- Normalization uses ImageNet mean and standard deviation
- FC7 feature size: `4096`
- Batch size for feature extraction: `64`

### Step-by-step flow
1. Define the image transform used by AlexNet.
2. Rebuild the DCNN architecture and load the saved checkpoint.
3. Remove the final classification layer so the model outputs FC7 features.
4. Normalize each segment channel by channel before feeding it to AlexNet.
5. Run feature extraction in batches to keep memory usage manageable.
6. Return one FC7 vector per segment.

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((227, 227), antialias=True),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def build_fc7_extractor(model_path):
    model = models.alexnet()
    num_ftrs = model.classifier[6].in_features
    model.classifier[6] = nn.Linear(num_ftrs, NUM_CLASSES)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.classifier = nn.Sequential(*list(model.classifier.children())[:5])
    model.eval()
    return model.to(device)

def preprocess_segment_for_alexnet(seg):
    img = seg.copy()
    for c in range(3):
        mn, mx = img[c].min(), img[c].max()
        img[c] = (img[c] - mn) / (mx - mn) if mx > mn else 0.0
    img = img.transpose(1, 2, 0)
    return transform(img).float()

def extract_fc7_batch(segments, extractor, batch_size=64):
    tensors = torch.stack([preprocess_segment_for_alexnet(s) for s in segments], dim=0)
    outputs = []
    with torch.no_grad():
        for i in range(0, len(tensors), batch_size):
            b = tensors[i:i+batch_size].to(device=device, dtype=torch.float32)
            outputs.append(extractor(b).cpu())
    return torch.cat(outputs, dim=0).numpy()

## Section 4. Grouping Segments into Utterance Sequences

### What is happening here?
The raw data is stored as individual segments, but emotion recognition over time only makes sense when we put those segments back in the order they appeared inside each utterance. This section groups segment files by utterance ID and prepares them as variable-length sequences.

### Why this matters
An utterance may have different emotions across time, so we cannot treat each segment in isolation if we want to study emotional flow. Grouping them into sequences gives the LSTM the full context it needs.

### Important parameters
- Input files: `X_*.npy`, `y_*.npy`, `utterance_ids_*.npy`
- Padding label value: `-100`
- Batch output shape for features: `[batch, max_len, feat_dim]`
- Batch output shape for labels: `[batch, max_len]`

### Step-by-step flow
1. Load the segment arrays for one split.
2. Group all segments that belong to the same utterance.
3. Extract FC7 features for every segment in that utterance.
4. Store the sequence of features, the sequence of labels, and the utterance ID.
5. In the collate function, pad shorter sequences so they can be batched together.
6. Mark padding labels with `-100` so the loss function ignores them.

In [4]:
def group_segments_by_utterance(X, y, utt_ids):
    grouped = defaultdict(lambda: {"segments": [], "labels": []})
    for seg, lbl, uid in zip(X, y, utt_ids):
        grouped[str(uid)]["segments"].append(seg)
        grouped[str(uid)]["labels"].append(int(lbl))
    return grouped

def build_sequences_from_split(dataset_dir, split_name, extractor):
    X = np.load(os.path.join(dataset_dir, f"X_{split_name}.npy"))
    y = np.load(os.path.join(dataset_dir, f"y_{split_name}.npy"))
    utt_ids = np.load(os.path.join(dataset_dir, f"utterance_ids_{split_name}.npy"))

    grouped = group_segments_by_utterance(X, y, utt_ids)
    seq_feats, seq_labels, seq_uids = [], [], []

    for uid, pack in grouped.items():
        seg_arr = np.array(pack["segments"])
        fc7 = extract_fc7_batch(seg_arr, extractor)
        seq_feats.append(torch.tensor(fc7, dtype=torch.float32))
        seq_labels.append(torch.tensor(pack["labels"], dtype=torch.long))
        seq_uids.append(uid)

    return seq_feats, seq_labels, seq_uids

class SequenceDataset(Dataset):
    def __init__(self, feats, labels, uids):
        self.feats = feats
        self.labels = labels
        self.uids = uids

    def __len__(self):
        return len(self.feats)

    def __getitem__(self, idx):
        return self.feats[idx], self.labels[idx], self.uids[idx]

def collate_sequences(batch):
    feats, labels, uids = zip(*batch)
    lengths = torch.tensor([len(x) for x in feats], dtype=torch.long)
    max_len = int(lengths.max().item())
    feat_dim = feats[0].shape[1]

    pad_feats = torch.zeros(len(batch), max_len, feat_dim, dtype=torch.float32)
    pad_labels = torch.full((len(batch), max_len), fill_value=-100, dtype=torch.long)

    for i, (f, l) in enumerate(zip(feats, labels)):
        T = len(f)
        pad_feats[i, :T] = f
        pad_labels[i, :T] = l

    return pad_feats, pad_labels, lengths, list(uids)

## Section 5. LSTM Model and Temporal Post-Processing

### What is happening here?
This section defines the actual sequence model. The LSTM reads a whole utterance one step at a time and predicts an emotion label for each segment. After that, we apply a small smoothing step so the predicted labels do not jump around too much from one segment to the next.

### Why this matters
Raw frame-by-frame predictions can be noisy. A smoothing step makes the output more human-like and helps neighboring segments with the same emotion stay together as a continuous block.

### Important parameters
- LSTM input size: `4096`
- Hidden size: `256`
- Number of layers: `1`
- Direction: bidirectional
- Dropout: only used when more than one layer is present
- Smoothing window: `SMOOTH_WINDOW` (default `5`)

### Step-by-step flow
1. Define a bidirectional LSTM that outputs one prediction per segment.
2. Map each hidden state to one of the emotion classes with a linear layer.
3. Smooth the predicted labels using majority vote over a sliding window.
4. Convert the smoothed labels into contiguous time intervals.
5. Compare predicted intervals with ground-truth intervals using IoU.

In [ ]:
class LSTMTagger(nn.Module):
    def __init__(self, input_dim=4096, hidden_dim=256, num_layers=1, num_classes=7, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        logits = self.head(out)
        return logits

def smooth_labels(seq, window=5):
    if window <= 1 or len(seq) == 0:
        return list(seq)
    half = window // 2
    out = []
    for i in range(len(seq)):
        left = max(0, i - half)
        right = min(len(seq), i + half + 1)
        votes = Counter(seq[left:right])
        out.append(votes.most_common(1)[0][0])
    return out

def sequence_to_intervals(labels, seg_hop_sec=SEG_HOP_SEC, seg_dur_sec=SEG_DUR_SEC):
    if len(labels) == 0:
        return []
    intervals = []
    start_idx = 0
    curr = labels[0]
    for i in range(1, len(labels)):
        if labels[i] != curr:
            s = start_idx * seg_hop_sec
            e = (i - 1) * seg_hop_sec + seg_dur_sec
            intervals.append((s, e, curr))
            start_idx = i
            curr = labels[i]
    s = start_idx * seg_hop_sec
    e = (len(labels) - 1) * seg_hop_sec + seg_dur_sec
    intervals.append((s, e, curr))
    return intervals

def interval_iou(a_start, a_end, b_start, b_end):
    inter = max(0.0, min(a_end, b_end) - max(a_start, b_start))
    union = max(a_end, b_end) - min(a_start, b_start)
    return (inter / union) if union > 0 else 0.0

def dominant_label(intervals):
    if not intervals:
        return 0
    dur = defaultdict(float)
    for s, e, l in intervals:
        dur[int(l)] += max(0.0, e - s)
    return sorted(dur.items(), key=lambda x: (-x[1], x[0]))[0][0]

## Section 6. Loading Timeline Labels and Measuring t-IoU

### What is happening here?
This section reads the timeline CSV files that were created when the combined emotion audio was generated. Those CSV files tell us when each emotion starts and ends in an utterance. We then compare the model prediction against those true emotion intervals.

### Why this matters
For multi-emotion utterances, a simple accuracy score is not enough. We need to know whether the model found the emotion in roughly the right time region. That is what t-IoU is used for.

### Important parameters
- Timeline root folder: `RAW_TIMELINE_ROOT`
- Matching rule: only compare intervals with the same emotion label
- Success threshold: `TIOU_THRESHOLD = 0.50`

### Step-by-step flow
1. Read each speaker’s timeline CSV file.
2. Build a lookup table from speaker and utterance ID to emotion intervals.
3. Sort the intervals by start time.
4. For each ground-truth interval, search all predicted intervals with the same label.
5. Keep the best overlap score for that emotion chunk.
6. Count it as correct if the best overlap is at least the threshold.
7. Average the results to get mean t-IoU and success rate.

In [6]:
def load_timeline_lookup(root_dir):
    lookup = defaultdict(dict)
    if not os.path.isdir(root_dir):
        return lookup

    for name in sorted(os.listdir(root_dir)):
        if not name.startswith("speaker_"):
            continue
        spk = name.split("_")[-1]
        csv_path = os.path.join(root_dir, name, f"speaker_{spk}_timeline.csv")
        if not os.path.exists(csv_path):
            continue

        with open(csv_path, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                uid = row.get("generated_file", "").strip()
                code = row.get("emotion_code", "").strip()
                if uid == "" or code not in EMOTION_CODE_MAP:
                    continue
                s = float(row.get("start_sec", 0.0))
                e = float(row.get("end_sec", 0.0))
                lookup[spk].setdefault(uid, []).append((s, e, EMOTION_CODE_MAP[code]))

    for spk in lookup:
        for uid in lookup[spk]:
            lookup[spk][uid].sort(key=lambda x: x[0])
    return lookup

def compute_interval_successes(pred_intervals, gt_intervals, threshold=TIOU_THRESHOLD):
    rows = []
    for gs, ge, gl in gt_intervals:
        best_iou = 0.0
        for ps, pe, pl in pred_intervals:
            if int(pl) != int(gl):
                continue
            best_iou = max(best_iou, interval_iou(gs, ge, ps, pe))
        rows.append({
            "label": int(gl),
            "best_iou": float(best_iou),
            "success": bool(best_iou >= threshold),
        })
    return rows

def compute_tiou(pred_intervals, gt_intervals, threshold=TIOU_THRESHOLD):
    if len(gt_intervals) == 0:
        return 0.0, 0.0, []

    rows = compute_interval_successes(pred_intervals, gt_intervals, threshold=threshold)
    best_ious = np.array([r["best_iou"] for r in rows], dtype=np.float32)
    mean_tiou = float(best_ious.mean()) if len(best_ious) > 0 else 0.0
    success_rate = float((best_ious >= threshold).mean()) if len(best_ious) > 0 else 0.0
    return mean_tiou, success_rate, rows

## Section 7. Training the LSTM and Collecting Evaluation Results

### What is happening here?
This section contains the training loop and the evaluation routine. The model is trained using cross-entropy loss on segment labels. After training, we evaluate it in several ways so that we can understand both the local segment predictions and the wider temporal behavior.

### Why this matters
A single metric rarely tells the whole story. Cross-entropy tells us how well the model is learning to classify segments during training. Segment accuracy tells us how often it gets the label right. t-IoU tells us whether it finds the correct emotion in the right time region.

### Important parameters
- Optimizer: `Adam`
- Learning rate: `1e-3`
- Maximum epochs: `20`
- Early stopping patience: `5`
- Loss function: `CrossEntropyLoss(ignore_index=-100)`
- Test loss accumulation: sum over valid segments only

### Step-by-step flow
1. Initialize the LSTM, optimizer, and cross-entropy loss.
2. Train for one epoch over the training loader.
3. Measure the validation loss after each epoch.
4. Save the best model based on validation loss.
5. Stop early if validation loss does not improve for several epochs.
6. During evaluation, predict one label for each segment.
7. Smooth the predictions, convert them to intervals, and compute t-IoU.
8. Collect segment accuracy, cross-entropy, and per-emotion detection signals for the final report.

In [15]:
def train_lstm_one_run(train_loader, val_loader, max_epochs=20, lr=1e-3, hidden_dim=256):
    model = LSTMTagger(hidden_dim=hidden_dim, num_classes=NUM_CLASSES).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(ignore_index=-100)

    best_val = float('inf')
    best_state = None
    patience = 5
    bad = 0
    history = []

    for epoch in range(max_epochs):
        model.train()
        train_losses = []
        for feats, labels, lengths, _ in train_loader:
            feats = feats.to(device)
            labels = labels.to(device)
            logits = model(feats)

            loss = criterion(logits.reshape(-1, NUM_CLASSES), labels.reshape(-1))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        with torch.no_grad():
            for feats, labels, lengths, _ in val_loader:
                feats = feats.to(device)
                labels = labels.to(device)
                logits = model(feats)
                val_loss = criterion(logits.reshape(-1, NUM_CLASSES), labels.reshape(-1))
                val_losses.append(val_loss.item())

        tr = float(np.mean(train_losses)) if train_losses else 0.0
        va = float(np.mean(val_losses)) if val_losses else 0.0
        history.append({
            "epoch": epoch + 1,
            "train_loss": tr,
            "val_loss": va,
        })
        print(f"Epoch {epoch+1:02d} | train_loss={tr:.4f} | val_loss={va:.4f}")

        if va < best_val:
            best_val = va
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                print("Early stopping triggered.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history

def evaluate_lstm_sequences(model, loader, timeline_lookup):
    model.eval()
    all_tiou = []
    all_success = []
    seg_true_all = []
    seg_pred_all = []
    interval_total = np.zeros(NUM_CLASSES, dtype=np.int64)
    interval_success = np.zeros(NUM_CLASSES, dtype=np.int64)
    interval_iou_sum = np.zeros(NUM_CLASSES, dtype=np.float64)
    ce_criterion = nn.CrossEntropyLoss(ignore_index=-100, reduction='sum')
    ce_sum = 0.0
    ce_tokens = 0

    with torch.no_grad():
        for feats, labels, lengths, uids in loader:
            feats = feats.to(device)
            labels_dev = labels.to(device)
            logits_t = model(feats)

            loss_sum = ce_criterion(logits_t.reshape(-1, NUM_CLASSES), labels_dev.reshape(-1))
            valid_tokens = int((labels_dev != -100).sum().item())
            ce_sum += float(loss_sum.item())
            ce_tokens += valid_tokens

            logits = logits_t.cpu().numpy()
            labels_np = labels.numpy()
            lengths_np = lengths.numpy()

            for b in range(len(uids)):
                T = int(lengths_np[b])
                uid = str(uids[b])

                pred_seq = np.argmax(logits[b, :T], axis=1).tolist()
                true_seq = labels_np[b, :T].tolist()

                pred_smoothed = smooth_labels(pred_seq, window=SMOOTH_WINDOW)
                pred_intervals = sequence_to_intervals(pred_smoothed)

                spk = uid[:2]
                gt_intervals = timeline_lookup.get(spk, {}).get(uid, None)
                if gt_intervals is None:
                    gt_intervals = sequence_to_intervals(true_seq)

                m_tiou, s_rate, tiou_rows = compute_tiou(pred_intervals, gt_intervals, threshold=TIOU_THRESHOLD)
                all_tiou.append(m_tiou)
                all_success.append(s_rate)

                for r in tiou_rows:
                    emo = int(r["label"])
                    interval_total[emo] += 1
                    interval_iou_sum[emo] += float(r["best_iou"])
                    if r["success"]:
                        interval_success[emo] += 1

                seg_true_all.extend(true_seq)
                seg_pred_all.extend(pred_smoothed[:T])

    seg_true_np = np.array(seg_true_all, dtype=np.int64)
    seg_pred_np = np.array(seg_pred_all, dtype=np.int64)
    seg_acc = accuracy_score(seg_true_np, seg_pred_np) if len(seg_true_np) > 0 else 0.0
    cross_entropy = float(ce_sum / ce_tokens) if ce_tokens > 0 else 0.0

    return {
        "y_true_seg": seg_true_np,
        "y_pred_seg": seg_pred_np,
        "segment_accuracy": float(seg_acc),
        "num_segments": int(len(seg_true_np)),
        "mean_tiou": float(np.mean(all_tiou)) if all_tiou else 0.0,
        "success_rate": float(np.mean(all_success)) if all_success else 0.0,
        "cross_entropy": cross_entropy,
        "ce_sum": float(ce_sum),
        "ce_tokens": int(ce_tokens),
        "interval_total": interval_total,
        "interval_success": interval_success,
        "interval_iou_sum": interval_iou_sum,
    }

## Section 8. Running the Full Experiment and Writing the Final Report

### What is happening here?
This is the last and most important section. It loops through the selected split or every LOSO fold, trains the LSTM, evaluates it, and then writes a full text report. The report includes the training progress for each epoch, the segment-level results, the temporal overlap results, and a per-emotion interval summary table.

### Why this matters
This is the part you will usually look at after a run finishes. It combines the training history and the final scores in one place so you do not need to manually inspect every cell output.

### Important parameters
- Batch size: `16`
- DCNN checkpoint naming pattern: `dcnn_{DATASET}.pth` or `dcnn_{DATASET}_{fold}.pth`
- Final report file: `classification_report_lstm_<dataset>.txt`
- Final outputs include per-segment accuracy, cross-entropy, mean t-IoU, success@threshold, and emotion-wise interval metrics (`tIoU`, `tIoU@thr`, `Succ`, `Total`)

### Step-by-step flow
1. Load the timeline lookup once.
2. For each split or fold, load the matching DCNN checkpoint.
3. Build train, validation, and test sequences from FC7 features.
4. Train the LSTM and store the epoch-by-epoch losses.
5. Evaluate on the test set and gather all metrics.
6. Combine fold results into global summaries.
7. Build the per-emotion table using interval-level temporal overlap.
8. Save everything to a text report and print a short summary on screen.

In [16]:
timeline_lookup = load_timeline_lookup(RAW_TIMELINE_ROOT)

all_seg_true = []
all_seg_pred = []
sum_interval_total = np.zeros(NUM_CLASSES, dtype=np.int64)
sum_interval_success = np.zeros(NUM_CLASSES, dtype=np.int64)
sum_interval_iou = np.zeros(NUM_CLASSES, dtype=np.float64)
fold_tious = []
fold_success = []
fold_seg_acc = []
fold_cross_entropy = []
fold_training_logs = []
total_ce_sum = 0.0
total_ce_tokens = 0

if SPLIT_MODE == "original":
    dcnn_path = os.path.join(MODEL_DIR, f"dcnn_{DATASET}.pth")
    if not os.path.exists(dcnn_path):
        raise FileNotFoundError(f"Missing DCNN checkpoint: {dcnn_path}")

    extractor = build_fc7_extractor(dcnn_path)
    tr_f, tr_l, tr_u = build_sequences_from_split(DATASET_PATH, "train", extractor)
    va_f, va_l, va_u = build_sequences_from_split(DATASET_PATH, "validation", extractor)
    te_f, te_l, te_u = build_sequences_from_split(DATASET_PATH, "test", extractor)

    train_loader = DataLoader(SequenceDataset(tr_f, tr_l, tr_u), batch_size=16, shuffle=True, collate_fn=collate_sequences)
    val_loader = DataLoader(SequenceDataset(va_f, va_l, va_u), batch_size=16, shuffle=False, collate_fn=collate_sequences)
    test_loader = DataLoader(SequenceDataset(te_f, te_l, te_u), batch_size=16, shuffle=False, collate_fn=collate_sequences)

    model, history = train_lstm_one_run(train_loader, val_loader)
    eval_out = evaluate_lstm_sequences(model, test_loader, timeline_lookup)

    fold_name = "original"
    fold_training_logs.append((fold_name, history))
    fold_tious.append(eval_out["mean_tiou"])
    fold_success.append(eval_out["success_rate"])
    fold_seg_acc.append(eval_out["segment_accuracy"])
    fold_cross_entropy.append(eval_out["cross_entropy"])
    total_ce_sum += eval_out["ce_sum"]
    total_ce_tokens += eval_out["ce_tokens"]

    all_seg_true.extend(eval_out["y_true_seg"].tolist())
    all_seg_pred.extend(eval_out["y_pred_seg"].tolist())
    sum_interval_total += eval_out["interval_total"]
    sum_interval_success += eval_out["interval_success"]
    sum_interval_iou += eval_out["interval_iou_sum"]

    print(f"Segment accuracy: {eval_out['segment_accuracy'] * 100:.2f}%")
    print(f"Cross-entropy (test): {eval_out['cross_entropy']:.4f}")
    print(f"Mean t-IoU: {eval_out['mean_tiou']:.4f}")
    print(f"t-IoU success@{TIOU_THRESHOLD:.2f}: {eval_out['success_rate'] * 100:.2f}%")

else:
    for fold_name in FOLD_NAMES:
        print('=' * 80)
        print(f"Fold: {fold_name}")
        fold_path = os.path.join(DATASET_PATH, fold_name)
        dcnn_path = os.path.join(MODEL_DIR, f"dcnn_{DATASET}_{fold_name}.pth")
        if not os.path.exists(dcnn_path):
            raise FileNotFoundError(f"Missing DCNN checkpoint: {dcnn_path}")

        extractor = build_fc7_extractor(dcnn_path)
        tr_f, tr_l, tr_u = build_sequences_from_split(fold_path, "train", extractor)
        va_f, va_l, va_u = build_sequences_from_split(fold_path, "validation", extractor)
        te_f, te_l, te_u = build_sequences_from_split(fold_path, "test", extractor)

        train_loader = DataLoader(SequenceDataset(tr_f, tr_l, tr_u), batch_size=16, shuffle=True, collate_fn=collate_sequences)
        val_loader = DataLoader(SequenceDataset(va_f, va_l, va_u), batch_size=16, shuffle=False, collate_fn=collate_sequences)
        test_loader = DataLoader(SequenceDataset(te_f, te_l, te_u), batch_size=16, shuffle=False, collate_fn=collate_sequences)

        model, history = train_lstm_one_run(train_loader, val_loader)
        eval_out = evaluate_lstm_sequences(model, test_loader, timeline_lookup)

        fold_training_logs.append((fold_name, history))
        fold_tious.append(eval_out["mean_tiou"])
        fold_success.append(eval_out["success_rate"])
        fold_seg_acc.append(eval_out["segment_accuracy"])
        fold_cross_entropy.append(eval_out["cross_entropy"])
        total_ce_sum += eval_out["ce_sum"]
        total_ce_tokens += eval_out["ce_tokens"]

        all_seg_true.extend(eval_out["y_true_seg"].tolist())
        all_seg_pred.extend(eval_out["y_pred_seg"].tolist())
        sum_interval_total += eval_out["interval_total"]
        sum_interval_success += eval_out["interval_success"]
        sum_interval_iou += eval_out["interval_iou_sum"]

        print(f"Fold segment accuracy: {eval_out['segment_accuracy'] * 100:.2f}%")
        print(f"Fold cross-entropy (test): {eval_out['cross_entropy']:.4f}")
        print(f"Fold mean t-IoU: {eval_out['mean_tiou']:.4f}")
        print(f"Fold t-IoU success@{TIOU_THRESHOLD:.2f}: {eval_out['success_rate'] * 100:.2f}%")

Fold: fold1
Epoch 01 | train_loss=1.2919 | val_loss=1.2697
Epoch 02 | train_loss=0.8869 | val_loss=1.1285
Epoch 03 | train_loss=0.7507 | val_loss=1.2235
Epoch 04 | train_loss=0.6884 | val_loss=1.2496
Epoch 05 | train_loss=0.6380 | val_loss=1.1468
Epoch 06 | train_loss=0.6070 | val_loss=1.1613
Epoch 07 | train_loss=0.5830 | val_loss=1.1188
Epoch 08 | train_loss=0.5600 | val_loss=1.2609
Epoch 09 | train_loss=0.5360 | val_loss=1.1123
Epoch 10 | train_loss=0.5425 | val_loss=1.1479
Epoch 11 | train_loss=0.5316 | val_loss=1.2445
Epoch 12 | train_loss=0.5067 | val_loss=1.1087
Epoch 13 | train_loss=0.5037 | val_loss=1.1715
Epoch 14 | train_loss=0.4953 | val_loss=1.2045
Epoch 15 | train_loss=0.4707 | val_loss=1.1058
Epoch 16 | train_loss=0.4560 | val_loss=1.1501
Epoch 17 | train_loss=0.4753 | val_loss=1.1190
Epoch 18 | train_loss=0.4579 | val_loss=1.0825
Epoch 19 | train_loss=0.4407 | val_loss=1.1694
Epoch 20 | train_loss=0.4308 | val_loss=1.1796
Fold segment accuracy: 67.82%
Fold cross-entropy

In [ ]:
y_true_seg_final = np.array(all_seg_true, dtype=np.int64)
y_pred_seg_final = np.array(all_seg_pred, dtype=np.int64)
global_seg_acc = accuracy_score(y_true_seg_final, y_pred_seg_final) if len(y_true_seg_final) > 0 else 0.0

emotion_names = [EMOTION_ENG_MAP.get(i, f"Class_{i}") for i in range(NUM_CLASSES)]
segment_report = (
    classification_report(
        y_true_seg_final,
        y_pred_seg_final,
        labels=np.arange(NUM_CLASSES),
        target_names=emotion_names,
        digits=4,
        zero_division=0,
    )
    if len(y_true_seg_final) > 0 else "No segments to report."
)

final_tiou = float(np.mean(fold_tious)) if fold_tious else 0.0
final_tiou_success = float(np.mean(fold_success)) if fold_success else 0.0
final_seg_acc_mean = float(np.mean(fold_seg_acc)) if fold_seg_acc else 0.0
final_ce_mean = float(np.mean(fold_cross_entropy)) if fold_cross_entropy else 0.0
global_ce = float(total_ce_sum / total_ce_tokens) if total_ce_tokens > 0 else 0.0
total_intervals_counted = int(sum_interval_total.sum())

emotion_rows = []
for emo in range(NUM_CLASSES):
    name = EMOTION_ENG_MAP.get(emo, str(emo))
    interval_success_val = int(sum_interval_success[emo])
    interval_total_val = int(sum_interval_total[emo])
    tiou_mean_val = float(sum_interval_iou[emo] / interval_total_val) if interval_total_val > 0 else 0.0
    tiou_thr_val = float(interval_success_val / interval_total_val) if interval_total_val > 0 else 0.0
    emotion_rows.append({
        "emotion_id": emo,
        "emotion": name,
        "tiou": tiou_mean_val,
        "tiou_thr": tiou_thr_val,
        "interval_success": interval_success_val,
        "interval_total": interval_total_val,
    })

table_header = (
    f"{'Emotion':<12} {'tIoU':>10} {'tIoU@thr':>10} {'Succ':>8} {'Total':>8}"
)
table_sep = '-' * len(table_header)

print('=' * 80)
print(f"Final global per-segment accuracy ({SPLIT_MODE}): {global_seg_acc * 100:.2f}%")
print(f"Final mean fold per-segment accuracy            : {final_seg_acc_mean * 100:.2f}%")
print(f"Final cross-entropy (global token-weighted)     : {global_ce:.4f}")
print(f"Final cross-entropy (mean across folds)         : {final_ce_mean:.4f}")
print(f"Final mean t-IoU                                : {final_tiou:.4f}")
print(f"Final t-IoU success@{TIOU_THRESHOLD:.2f}        : {final_tiou_success * 100:.2f}%")
print(f"Final Section B total GT intervals counted      : {total_intervals_counted}")

print("\nSection A. Segment-level classification:")
print(segment_report)

print("Section B. Interval-level temporal detection by emotion (aggregated over all test utterances):")
print(table_header)
print(table_sep)
for row in emotion_rows:
    print(
        f"{row['emotion']:<12} "
        f"{row['tiou']:>10.3f} {row['tiou_thr']:>10.3f} "
        f"{row['interval_success']:>8d} {row['interval_total']:>8d}"
    )

print("Interpretation:")
print("    - tIoU     : mean best-overlap IoU per ground-truth interval for that emotion.")
print("    - tIoU@thr : fraction of intervals with IoU >= threshold for that emotion.")
print("    - Succ     : count of successful intervals for that emotion.")
print("    - Total    : count of all ground-truth intervals for that emotion.")

report_path = os.path.join(RESULTS_DIR, f"classification_report_lstm_{DATASET}.txt")
with open(report_path, "w") as f:
    f.write(f"{f'Dataset                                ':<38} : {DATASET}\n")
    f.write(f"{f'Split mode                             ':<38} : {SPLIT_MODE}\n")
    f.write(f"{f'Global Mean per-segment accuracy       ':<38} : {global_seg_acc * 100:.2f}%\n")
    f.write(f"{f'Fold Mean per-segment accuracy         ':<38} : {final_seg_acc_mean * 100:.2f}%\n")
    f.write(f"{f'Global Mean Cross-entropy              ':<38} : {global_ce:.4f}\n")
    f.write(f"{f'Fold Mean Cross-entropy                ':<38} : {final_ce_mean:.4f}\n")
    f.write(f"{f't-IoU (Fold Mean)                      ':<38} : {final_tiou:.4f}\n")
    f.write(f"{f't-IoU success@{TIOU_THRESHOLD:.2f}     ':<38} : {final_tiou_success * 100:.2f}%\n")
    f.write(f"{f'Section B total GT intervals counted   ':<38} : {total_intervals_counted}\n\n")

    f.write("Note:\n")
    f.write("  Global metrics are weighted by fold size while fold metrics are unweighted across folds.\n")
    f.write("  They match only when all folds have the same number of segments.\n")

    f.write("\nSegment-level classification report:\n")
    f.write(segment_report)

    f.write("\nClass index mapping:\n")
    for idx, name in enumerate(emotion_names):
        f.write(f"{idx}: {name}\n")

    f.write("\nSection B. Interval-level temporal detection by emotion (aggregated over all test utterances):\n")
    f.write(table_header + "\n")
    f.write(table_sep + "\n")
    for row in emotion_rows:
        f.write(
            f"{row['emotion']:<12} "
            f"{row['tiou']:>10.3f} {row['tiou_thr']:>10.3f} "
            f"{row['interval_success']:>8d} {row['interval_total']:>8d}\n"
        )

    f.write("\nInterpretation:\n")
    f.write("    - tIoU     : mean best-overlap IoU per ground-truth interval for that emotion.\n")
    f.write("    - tIoU@thr : fraction of intervals with IoU >= threshold for that emotion.\n")
    f.write("    - Succ     : count of successful intervals for that emotion.\n")
    f.write("    - Total    : count of all ground-truth intervals for that emotion.\n")

    f.write("\nTraining progress per split/fold (epoch-wise):\n")
    for fold_name, history in fold_training_logs:
        f.write(f"\n[{fold_name}]\n")
        for row in history:
            f.write(
                f"Epoch {row['epoch']:02d} | train_loss={row['train_loss']:.4f} | val_loss={row['val_loss']:.4f}\n"
            )
print(f"Saved LSTM report to: {report_path}")

Final global per-segment accuracy (loso): 69.33%
Final mean fold per-segment accuracy            : 69.42%
Final cross-entropy (global token-weighted)     : 0.9470
Final cross-entropy (mean across folds)         : 0.9432
Final mean t-IoU                                : 0.5733
Final t-IoU success@0.50        : 62.58%
Final Section B total GT intervals counted      : 1977

Section A. Segment-level classification:
              precision    recall  f1-score   support

       Anger     0.7242    0.8811    0.7950      3769
     Boredom     0.6609    0.5610    0.6069      2515
        Fear     0.6691    0.5565    0.6077      1937
   Happiness     0.5190    0.4228    0.4660      2001
     Sadness     0.8373    0.9260    0.8794      3068
     Disgust     0.6508    0.6303    0.6404      1585
     Neutral     0.6166    0.6125    0.6146      2266

    accuracy                         0.6933     17141
   macro avg     0.6683    0.6558    0.6586     17141
weighted avg     0.6840    0.6933    0.6848